# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [2]:
#Initial imports

import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json
import pprint as pp
from IPython.display import display

from DbConnector import DbConnector

In [ ]:
# Import datasets
DS_DIR = "movies_ds/"

credits_df = pd.read_csv(DS_DIR + "credits.csv")
keywords_df = pd.read_csv(DS_DIR + "keywords.csv")
links_small_df = pd.read_csv(DS_DIR + "links_small.csv")
links_df = pd.read_csv(DS_DIR + "links.csv")
movies_metadata_df = pd.read_csv(DS_DIR + "movies_metadata.csv")
ratings_small_df = pd.read_csv(DS_DIR + "ratings_small.csv")
ratings_df = pd.read_csv(DS_DIR + "ratings.csv")

In [ ]:
print("Credits: " + str(credits_df.shape))
display(credits_df.head())

print("Keywords: " + str(keywords_df.shape))
display(keywords_df.head())

print("Links_small: " + str(links_small_df.shape))
display(links_small_df.head())

print("Links: " + str(links_df.shape))
display(links_df.head())

print("Movies_metadata: " + str(movies_metadata_df.shape))
display(movies_metadata_df.head())

print("Ratings_small: " + str(ratings_small_df.shape))
display(ratings_small_df.head())

print("Ratings: " + str(ratings_df.shape))
display(ratings_df.head())

In [ ]:
# Cleaning of the data
credits_df_clean = credits_df.copy()
keywords_df_clean = keywords_df.copy()
links_small_df_clean = links_small_df.copy()
links_df_clean = links_df.copy()
movies_metadata_df_clean = movies_metadata_df.copy()
ratings_small_df_clean = ratings_small_df.copy()
ratings_df_clean = ratings_df.copy()

# Getting rid of duplicates (movies with the same id)
movies_metadata_df_clean = movies_metadata_df_clean.drop_duplicates(subset='id')

- About linear reg for budget based on ratings
- Redundancy for the database structure

# Movies
- **id** (tmdbId)
- adult
- collections
    - name (remove id)
- keywords
    - name (remove id)
- genres
    - name (remove id)
- budget
- original_language
- overview (task 7)
- production_companies
    - name (remove id)
- production_countries
    - iso_3166_1 (remove name)
- release_date
- revenue
- runtime
- spoken_languages (dont need, can maybe remove)
    - iso_639_1 (remove name)
- status
- tagline
- title
- video
- vote_average
- vote_count
- cast
    - contributor_id
    - order
- crew
    - contributor_id
    - job

# Contributors
- **contributor_id**
- name (all people with more names have english and native so could just do english)
- gender (all people with more genders have 0 and either 1 or 2, so prob fine to just ignore 0 here)
- roles
    - tmdbId
    - character
- jobs
    - tmdbId
    - department
    - job

# Ratings
- userId
- tmdbId (not movieId)
- rating
- timestamp

Considered storing ratings on users...

Because average on metadata and the nature of the exercises. Also, since this is static data analysis, we don't have to account for the growing documents. Some fields are removed because they relate to external resources, and this is an internal analysis. Some have also been deleted because they have been deemed unnecessary for the broad direction of this analysis.

In [ ]:
movies_metadata_df["budget"] = pd.to_numeric(
    movies_metadata_df["budget"], errors="coerce"
)
movies_metadata_df["budget"] = movies_metadata_df["budget"].astype("Int64")

movies_metadata_df.info()

In [ ]:
import pandas as pd
import ast

def parse_credits(value):
    if pd.isna(value) or value == "":
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

people = []

for _, row in credits_df.iterrows():
    for column in ["cast", "crew"]:
        for person in parse_credits(row[column]):
            people.append({
                "id": person.get("id"),
                "name": person.get("name"),
                "gender": person.get("gender")
            })

people_df = pd.DataFrame(people)

# Remove missing IDs and genders
people_df = people_df.dropna(subset=["id", "gender"])

# Find each person's distinct gender values
gender_details = (
    people_df.groupby(["id", "name"])["gender"]
    .agg(lambda x: sorted(set(x)))
    .reset_index(name="genders")
)

# Keep only people with multiple gender values
gender_details = gender_details[
    gender_details["genders"].apply(len) > 1
]

print(gender_details.to_string(index=False))

In [ ]:
import pandas as pd
import ast

def parse_credits(value):
    if pd.isna(value) or value == "":
        return []
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

people = []

for _, row in credits_df.iterrows():
    for column in ["cast", "crew"]:
        for person in parse_credits(row[column]):
            people.append({
                "id": person.get("id"),
                "name": person.get("name"),
                "role_type": column
            })

people_df = pd.DataFrame(people)

# Remove missing IDs and names
people_df = people_df.dropna(subset=["id", "name"])

# Find people with more than one distinct name
name_counts = (
    people_df.groupby("id")["name"]
    .nunique()
    .reset_index(name="number_of_names")
)

multiple_names = name_counts[name_counts["number_of_names"] > 1]

# Show all names associated with each ID
name_details = (
    people_df[people_df["id"].isin(multiple_names["id"])]
    .groupby("id")["name"]
    .agg(lambda names: sorted(set(names)))
    .reset_index(name="names")
)

name_details["number_of_names"] = name_details["names"].apply(len)

print(name_details.to_string(index=False))

In [ ]:

print("Total movies:", len(movies_metadata_df))

print("Budget = 0:", (movies_metadata_df["budget"] == 0).sum())

for movie in movies_metadata_df["budget"]:
    if movie % 1 != 0:
        print(movie)

print("Budget missing:", movies_metadata_df["budget"].isna().sum())

In [ ]:
df_budget_cleaning = movies_metadata_df[
    (movies_metadata_df["rating_count"].notna())
].copy()

ratingCounts = ratings_df["movieId"].value_counts(dropna=False).reset_index()
ratingCounts.columns = ["movieId", "count"]

df = movies_metadata_df[
    (movies_metadata_df["rating_count"].notna())
].copy()

In [ ]:
from datetime import datetime

def timestamp_to_datetime(timestamp):
    """
    In: timestamp (number of seconds since January 1st, 1970)
        typically items from the 'timestamp' column in the ratings_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.fromtimestamp(timestamp)

def date_to_datetime(date):
    """
    In: date following this format: YYYY-MM-DD as a string
        typically items from the 'release_date' column in the movies_metadata_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.strptime(date, "%Y-%m-%d")

# Getting rid of reviews posted before the movie release date
nb_ratings_deleted = 0
for index, row in ratings_df_clean.iterrows():
    rating_datetime = timestamp_to_datetime(row['timestamp'])
    movie_id = str(int(row['movieId']))
    movie = movies_metadata_df_clean.loc[movies_metadata_df_clean['id'] == movie_id]
    if not movie.empty:
        movie = movie.iloc[0]
        movie_release_datetime = date_to_datetime(str(movie['release_date']))
        if rating_datetime < movie_release_datetime:
            ratings_df_clean.drop(index)
            nb_ratings_deleted += 1

In [ ]:
# Check rating range
print("\n--- Full Ratings Dataset ---")
print(f"Min rating: {ratings_df_clean['rating'].min()}")
print(f"Max rating: {ratings_df_clean['rating'].max()}")

# Unique keywords: some rows have empty lists, which is not null but still missing data
unique_keywords = keywords_df_clean['id'].nunique()
print(f"\nUnique keywords: {unique_keywords}")

In [ ]:

print("Credits: \n", credits_df_clean.isnull().sum())
print("\nKeywords: \n", keywords_df_clean.isnull().sum())
print("\nLinks: \n", links_df_clean.isnull().sum())
print("\nRatings: \n", ratings_df_clean.isnull().sum())
print("\nMetadata: \n", movies_metadata_df_clean.isnull().sum())

In [ ]:
# Sample of movies_metadata_df_clean null values in title
print("\nLinks: \n")
print(movies_metadata_df_clean.loc[
    movies_metadata_df_clean["title"].isnull(),
    ["original_title", "title"],
])

In [ ]:
bad_title = movies_metadata_df_clean[movies_metadata_df_clean["title"].isnull()]
print(bad_title[["id", "adult", "original_title", "release_date", "vote_count"]])

ids_numeric = pd.to_numeric(movies_metadata_df_clean["id"], errors="coerce")
print("Non-numeric ids:", movies_metadata_df_clean.loc[ids_numeric.isnull(), "id"].tolist())

# 'adult' should only be True/False
print(movies_metadata_df_clean["adult"].value_counts().head())

In [ ]:
null_title_idx = movies_metadata_df_clean.index[movies_metadata_df_clean["title"].isnull()]
is_tail = pd.to_numeric(movies_metadata_df_clean.loc[null_title_idx, "id"], errors="coerce").isnull()
tail_idx = null_title_idx[is_tail]
head_idx = null_title_idx[~is_tail]

assert len(head_idx) == len(tail_idx) == 3
assert (head_idx + 1 == tail_idx).all()

recovered_dates = movies_metadata_df_clean.loc[tail_idx, "id"].values
assert pd.to_datetime(recovered_dates, format="%Y-%m-%d", errors="coerce").notnull().all()
movies_metadata_df_clean.loc[head_idx, "release_date"] = recovered_dates

movies_metadata_df_clean.loc[head_idx, "title"] = movies_metadata_df_clean.loc[head_idx, "original_title"]

# Drop the malformed tail rows
movies_metadata_df_clean = movies_metadata_df_clean.drop(index=tail_idx)

# Verify
print("Null titles remaining:", movies_metadata_df_clean["title"].isnull().sum())
print("Non-numeric ids remaining:", pd.to_numeric(movies_metadata_df_clean["id"], errors="coerce").isnull().sum())
print(movies_metadata_df_clean.loc[head_idx, ["id", "original_title", "title", "release_date"]])

In [ ]:
# movies_metadata sample of ids
print("\nSample of movies_metadata_df ids:")
print(movies_metadata_df_clean.loc[head_idx, ["id", "imdb_id"]])

print("\nClean imdb_id in movies_metadata_df")
# Check how many start with 'tt'
sample_imdb = movies_metadata_df_clean['imdb_id'].dropna().astype(str)
starts_with_tt = sample_imdb.str.startswith('tt').all()
print(f"All non-null imdb_id start with 'tt': {starts_with_tt}")

# Remove 'tt' prefix and convert remainder to nullable integer
movies_metadata_df_clean['imdb_id_clean'] = movies_metadata_df_clean['imdb_id'].astype(str).str.replace('tt', '', regex=False)
movies_metadata_df_clean['imdb_id_clean'] = pd.to_numeric(movies_metadata_df_clean['imdb_id_clean'], errors='coerce').astype('Int64')

print("Sample cleaned imdb_id values:")
print(movies_metadata_df_clean[['imdb_id', 'imdb_id_clean']].head(3))

# Drop the old imdb_id column and rename the clean version
movies_metadata_df_clean = movies_metadata_df_clean.drop(columns=['imdb_id'])
movies_metadata_df_clean = movies_metadata_df_clean.rename(columns={'imdb_id_clean': 'imdb_id'})

In [ ]:
from pymongo import MongoClient, version

class DbConnector:
    """
    Connects to the MongoDB server on the Ubuntu virtual machine.
    Connector needs HOST, USER and PASSWORD to connect.

    Example:
    HOST = "tdt4225-00.idi.ntnu.no" // Your server IP address/domain name
    USER = "testuser" // This is the user you created and added privileges for
    PASSWORD = "test123" // The password you set for said user
    """

    def __init__(self,
                 DATABASE='my_db',
                 PORT=27018,
                 HOST="localhost",
                 USER="groupuser",
                 PASSWORD="WaterisGood02"):
        uri = "mongodb://%s:%s@%s:%d/%s?authSource=%s" % (USER, PASSWORD, HOST, PORT, DATABASE, DATABASE)
        # Connect to the databases
        try:
            self.client = MongoClient(uri)
            self.db = self.client[DATABASE]
        except Exception as e:
            print("ERROR: Failed to connect to db:", e)

        # get database information
        print("You are connected to the database:", self.db.name)
        print("-----------------------------------------------\n")

    def close_connection(self):
        # close the cursor
        # close the DB connection
        self.client.close()
        print("\n-----------------------------------------------")
        print("Connection to %s-db is closed" % self.db.name)

In [ ]:
# Prepare data in structure
def parse_nested(value):
    if not isinstance(value, str) or not value.strip():
        return []

    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []

In [ ]:
# Build lookup dictionaries
credits_lookup = (
    credits_df.dropna(subset=["id"])
    .drop_duplicates("id")
    .set_index("id")
    .to_dict("index")
)

keywords_lookup = (
    keywords_df.dropna(subset=["id"])
    .drop_duplicates("id")
    .set_index("id")
    .to_dict("index")
)

movie_fields = [
    "adult",
    "belongs_to_collection",
    "genres",
    "budget",
    "original_language",
    "overview",
    "release_date",
    "revenue",
    "runtime",
    "status",
    "tagline",
    "title",
    "video",
    "vote_average",
    "vote_count",
    "production_companies",
    "production_countries",
    "spoken_languages",
]

movie_docs = []

for movie in movies_metadata_df_clean.to_dict("records"):
    tmdb_id = movie["id"]

    credits = credits_lookup.get(tmdb_id, {})
    keywords = keywords_lookup.get(tmdb_id, {})

    doc = {
        "_id": tmdb_id,
        **{
            field: movie.get(field)
            for field in movie_fields
            if field in movie
        },
    }

    # Parse nested metadata
    collection = doc.get("belongs_to_collection")
    if isinstance(collection, dict):
        doc["collection"] = {
            "name": collection.get("name")
        }
    else:
        doc["collection"] = None

    doc.pop("belongs_to_collection", None)

    for field in [
        "genres",
        "production_companies",
        "production_countries",
        "spoken_languages",
    ]:
        parsed = parse_nested(doc.get(field))
        doc[field] = parsed

    doc["keywords"] = parse_nested(keywords.get("keywords"))

    # Keep useful cast fields
    doc["cast"] = [
        {
            "contributor_id": person.get("id"),
            "order": person.get("order"),
            "character": person.get("character"),
        }
        for person in parse_nested(credits.get("cast"))
    ]

    # Keep useful crew fields
    doc["crew"] = [
        {
            "contributor_id": person.get("id"),
            "job": person.get("job"),
            "department": person.get("department"),
        }
        for person in parse_nested(credits.get("crew"))
    ]

    # Replace pandas missing values with None
    doc = {
        key: (None if pd.isna(value) else value)
        if not isinstance(value, (list, dict))
        else value
        for key, value in doc.items()
    }

    movie_docs.append(doc)

print(f"Prepared {len(movie_docs):,} movie documents")

In [ ]:
contributors = {}

for movie in movie_docs:
    tmdb_id = movie["_id"]

    # Process cast
    for person in parse_nested(
        credits_lookup.get(tmdb_id, {}).get("cast")
    ):
        person_id = person.get("id")

        if person_id is None:
            continue

        contributor = contributors.setdefault(
            person_id,
            {
                "_id": person_id,
                "name": person.get("name"),
                "gender": person.get("gender", 0),
                "roles": [],
                "jobs": [],
            }
        )

        contributor["roles"].append({
            "tmdbId": tmdb_id,
            "character": person.get("character"),
        })

    # Process crew
    for person in parse_nested(
        credits_lookup.get(tmdb_id, {}).get("crew")
    ):
        person_id = person.get("id")

        if person_id is None:
            continue

        contributor = contributors.setdefault(
            person_id,
            {
                "_id": person_id,
                "name": person.get("name"),
                "gender": person.get("gender", 0),
                "roles": [],
                "jobs": [],
            }
        )

        contributor["jobs"].append({
            "tmdbId": tmdb_id,
            "department": person.get("department"),
            "job": person.get("job"),
        })

contributor_docs = list(contributors.values())

print(f"Prepared {len(contributor_docs):,} contributor documents")

In [ ]:
links_df["tmdbId"] = pd.to_numeric(
    links_df["tmdbId"], errors="coerce"
)

links_df = links_df.dropna(subset=["tmdbId"]).copy()
links_df["tmdbId"] = links_df["tmdbId"].astype("int64")

movie_id_map = links_df.set_index("movieId")["tmdbId"].to_dict()

# Movies
- **id** (tmdbId)
- adult
- collections
    - name (remove id)
- keywords
    - name (remove id)
- genres
    - name (remove id)
- budget
- original_language
- overview (task 7)
- production_companies
    - name (remove id)
- production_countries
    - iso_3166_1 (remove name)
- release_date
- revenue
- runtime
- spoken_languages (dont need, can maybe remove)
    - iso_639_1 (remove name)
- status
- tagline
- title
- video
- vote_average
- vote_count
- cast
    - contributor_id
    - order
- crew
    - contributor_id
    - job

# Contributors
- **contributor_id**
- name (all people with more names have english and native so could just do english)
- gender (all people with more genders have 0 and either 1 or 2, so prob fine to just ignore 0 here)
- roles
    - tmdbId
    - character
- jobs
    - tmdbId
    - department
    - job

# Raters
- userId
- tmdbId (not movieId)
- rating
- timestamp

Because average on metadata and the nature of the exercises. Also, since this is static data analysis, we don't have to account for the growing documents.

In [ ]:
from pprint import pprint 
from DbConnector import DbConnector

class Program:

    def __init__(self):
        self.connection = DbConnector()
        self.client = self.connection.client
        self.db = self.connection.db

    def create_coll(self, collection_name):
        collection = self.db.create_collection(collection_name)    
        print('Created collection: ', collection)

    def insert_documents(self, docs, collection_name):
        collection = self.db[collection_name]
        collection.insert_many(docs)

    def insert_with_progress(self, collection_name, docs, batch_size=1000):
        total = len(docs)
        collection = self.db[collection_name]

        for start in range(0, total, batch_size):
            batch = docs[start:start + batch_size]
            collection.insert_many(batch)

            inserted = min(start + len(batch), total)
            print(f"{collection.name}: {inserted:,} / {total:,} inserted")

    def insert_ratings(self, batch_size=100_000):
        collection = self.db["Ratings"]
        for chunk in pd.read_csv("ratings.csv", batch_size):
            chunk["tmdbId"] = chunk["movieId"].map(movie_id_map)

            # Exclude ratings whose MovieLens IDs have no TMDB mapping
            chunk = chunk.dropna(subset=["tmdbId"])
            chunk["tmdbId"] = chunk["tmdbId"].astype("int64")

            rating_docs = chunk[
                ["userId", "tmdbId", "rating", "timestamp"]
            ].to_dict("records")

            if rating_docs:
                collection.insert_many(rating_docs, ordered=False)

        print("Ratings inserted.")
        
    def fetch_documents(self, collection_name):
        collection = self.db[collection_name]
        documents = collection.find({})
        for doc in documents: 
            pprint(doc)
        

    def drop_coll(self, collection_name):
        collection = self.db[collection_name]
        collection.drop()

    def show_coll(self):
        collections = self.db.list_collection_names()
        print(collections)


def main():
    program = None
    try:
        program = Program()
        print("Its up")
        program.show_coll()
        print("Should have shown coll")
        #program.drop_coll(collection_name="Movies")
        #program.drop_coll(collection_name="Contributors")
        #program.drop_coll(collection_name="Ratings")

        program.create_coll(collection_name="Movies")
        program.create_coll(collection_name="Contributors")
        program.create_coll(collection_name="Ratings")
        program.show_coll()

        #program.insert_documents(movie_docs, collection_name="Movies")

        #program.insert_documents(contributor_docs, collection_name="Contributors")

        #program.insert_ratings()

        program.fetch_documents(collection_name="Person")
        program.drop_coll(collection_name="Person")

        # Check that the table is dropped
        program.show_coll()
    except Exception as e:
        print("ERROR: Failed to use database:", e)
    finally:
        if program:
            program.connection.close_connection()


if __name__ == '__main__':
    main()
    print("Done")